# EDA 공통 템플릿 — [데이터셋명]

> 이 파일을 **EDA_데이터셋명.ipynb**로 복사해 담당 데이터 **하나만** 분석하세요. CSV 결과는 로컬 `data/processed`에 저장되며 GitHub에서는 무시됩니다.

**기록 필수**: 담당자 / 원본 기관·출처 / 데이터 기준일 / 파일명 / 행 단위 / 분석 목적 / 공간 및 시간 범위 / 컬럼 정의서.

목표: 원본 품질 확인 → 근거 기반 정제 → 통계·공간 분석 → CSV 재검증 → 프로젝트/ML 활용성 평가.

**주의:** 원본을 변경하지 않고 실제 흡연 발생 라벨을 임의 생성하지 않습니다. 공통 250m 격자와 CRS 확정 전에는 격자 ID를 임의 생성하지 마세요.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 저장소 루트 기준으로 설정 (VS Code 커널의 작업 디렉터리 확인)
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name in {"01_eda", "notebooks"}:
    REPO_ROOT = next((p for p in [REPO_ROOT, *REPO_ROOT.parents] if (p / "README.md").is_file() and (p / "data").is_dir()), REPO_ROOT)
RAW_RELATIVE_PATH = None  # 예: "data/raw/02_4_REST데이터/서울특별시_양천구_흡연시설 현황_20260828.csv"
DATASET_KEY = "dataset_name"  # 예: bus_stops (영문/숫자/밑줄)
CSV_ENCODING = "utf-8-sig"  # 입력 파일 인코딩이 다르면 바꾸세요.
SHEET_NAME = 0             # Excel 시트 번호/이름
OUTPUT_DIR = REPO_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("프로젝트 루트:", REPO_ROOT)
print("데이터 경로 설정:", RAW_RELATIVE_PATH)

## 1. 원본 로딩과 구조
- 원본 파일이 저장소 `data/raw/`에 있는지 확인합니다.
- ZIP 파일은 내부 파일명과 형식을 먼저 확인하고 필요한 파일만 별도 추출한 후 상대경로로 지정합니다.
- Excel/CSV는 인코딩·자료형·식별자의 0으로 시작하는 값 등을 검토합니다.

In [ ]:
df = None
if RAW_RELATIVE_PATH is None:
    print("RAW_RELATIVE_PATH를 지정한 뒤 다시 실행하세요. 샘플·가짜 데이터를 생성하지 않습니다.")
else:
    source_path = REPO_ROOT / RAW_RELATIVE_PATH
    if not source_path.is_file():
        raise FileNotFoundError(f"원본을 찾지 못했습니다: {source_path}")
    suffix = source_path.suffix.lower()
    if suffix == ".csv":
        df = pd.read_csv(source_path, encoding=CSV_ENCODING, low_memory=False)
    elif suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(source_path, sheet_name=SHEET_NAME)
    else:
        raise ValueError(f"이 템플릿에서 지원하지 않는 형식: {suffix}; ZIP은 내부 데이터 추출 후 분석")
    print("shape:", df.shape)
    display(df.head(10))
    display(pd.DataFrame({"dtype": df.dtypes.astype(str), "non_null": df.notna().sum(), "nunique": df.nunique(dropna=False)}))

## 2. 데이터 품질
결측, 중복, 자료형, 수치 분포, 지역·시간·좌표 유효성 및 이상치 판단 기준을 분석 결과와 함께 작성하세요.

In [ ]:
quality = pd.DataFrame()
if df is not None:
    quality = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "missing_count": df.isna().sum().values,
        "missing_pct": (df.isna().mean()*100).round(3).values,
        "unique_count": df.nunique(dropna=False).values
    })
    display(quality.sort_values("missing_pct", ascending=False))
    print("완전 동일 행 수:", int(df.duplicated().sum()))
    display(df.describe(include="all").T)
else:
    print("데이터를 불러온 후 실행하세요.")

## 3. EDA와 시각화
- 데이터에 적합한 수치·범주·시간 분석을 수행하세요.
- 위치 데이터는 CRS, 좌표 유효성, 주소의 활용 가능성을 확인하세요.
- 그래프마다 **관찰한 사실**, **해석**, **한계**를 Markdown으로 작성하세요.

In [ ]:
if df is not None:
    num_cols = df.select_dtypes(include="number").columns.tolist()
    if num_cols:
        col = num_cols[0]
        ax = pd.to_numeric(df[col], errors="coerce").dropna().hist(bins=30, figsize=(7,4))
        ax.set(title=f"{col} distribution", xlabel=col, ylabel="count")
        plt.tight_layout()
        plt.show()
    else:
        print("수치형 컬럼 없음: 범주형 빈도 분석 등을 추가하세요.")

## 4. 정제 정책 및 전처리
**데이터셋별 판단 필수.** 임의로 결측치/이상치/중복을 제거하지 마세요. 아래 기본 코드는 원본을 보존하는 복사본만 만듭니다. 실제 변환에는 전후 건수 및 기준을 기록하세요.

In [ ]:
clean = None
if df is not None:
    clean = df.copy(deep=True)
    # TODO: 검증된 규칙만 추가하세요. 예: 식별자 공백 정리, 날짜 변환, 확실한 중복 제거 등
    print({"rows_before":len(df), "rows_after":len(clean), "columns_before":len(df.columns), "columns_after":len(clean.columns)})
    display(clean.head())

## 5. 결과 CSV 저장 및 재검증
- 반드시 **clean.csv**, **summary.csv**를 생성합니다.
- summary는 실제 계산한 컬럼별 품질 수치를 포함합니다.
- features는 정당한 파생변수가 실제로 생성된 경우에만 저장합니다.
- 영문 `DATASET_KEY`는 파일명 충돌이 없도록 팀에서 정하세요.

In [ ]:
if clean is not None:
    clean_path = OUTPUT_DIR / f"{DATASET_KEY}_clean.csv"
    summary_path = OUTPUT_DIR / f"{DATASET_KEY}_summary.csv"
    clean.to_csv(clean_path, encoding="utf-8-sig", index=False)
    quality.to_csv(summary_path, encoding="utf-8-sig", index=False)
    restored = pd.read_csv(clean_path, encoding="utf-8-sig", low_memory=False)
    assert restored.shape == clean.shape, "저장/재읽기 shape 불일치"
    assert restored.columns.tolist() == clean.columns.tolist(), "저장/재읽기 컬럼 불일치"
    print("저장 완료:", clean_path, summary_path)
    print("주의: 재읽기 자료형/선행 0 등도 확인하세요.")
else:
    print("먼저 유효한 원본을 불러오고 clean 데이터를 만든 뒤 실행하세요.")

## 6. 파생변수와 프로젝트 적용 계획

아래 표를 **실제 컬럼과 계산식**으로 채우세요.

| 변수 | 원천 컬럼 | 계산식 | 공간/시간 단위 | 모델 적용 | 한계 |
|---|---|---|---|---|---|
| (직접 작성) | | | | | |

- 250m 격자 공간 조인 가능 여부(공통 격자·CRS 미확정 시 보류)
- 시간 단위 및 공간 단위가 다른 자료와 결합할 때 필요한 키
- 통계 분석/군집화/지도학습 중 타당한 방법과 이유
- 지도학습 정답 라벨 보유 여부 및 누수/편향·성능 검증 필요사항
- 서비스 적용: 위험요인 특성 → 격자/도로 링크 가중치 → Dijkstra/A* 중 **검증 가능한 연결 단계**
- 관측값과 추정/가설을 명확히 구분

### features.csv 생성 (선택)
데이터에서 의미 있는 파생변수를 계산했다면 `features` 데이터프레임을 구성하고, 계산식과 단위를 설명한 후 아래 코드를 실행하세요.

In [ ]:
features = None  # TODO: 실제 근거 있는 파생변수 데이터프레임을 지정
if isinstance(features, pd.DataFrame) and not features.empty:
    feature_path = OUTPUT_DIR / f"{DATASET_KEY}_features.csv"
    features.to_csv(feature_path, encoding="utf-8-sig", index=False)
    check = pd.read_csv(feature_path, encoding="utf-8-sig", low_memory=False)
    assert check.shape == features.shape
    print("저장 완료:", feature_path)
else:
    print("생성할 파생변수 없음: 이유를 Notebook 결론에 적어주세요.")

## 7. 결론 및 팀 전달
- 핵심 발견 3~5개 (수치·그래프 근거)
- 전처리 전후 비교와 품질 문제
- 프로젝트에 활용할 변수·가능한 분석기법·예상 적용 단계
- 필요한 추가 데이터/검증 계획
- **최종 판정**: 바로 사용 / 추가 전처리 후 사용 / 추가 데이터 필요 / 사용 보류
- 저장된 CSV 파일 경로(Drive 공유 위치 포함), Python/라이브러리 환경, 재실행 방법
- PR에는 Notebook + 소형 분석 요약만 올리고 data/raw·data/processed는 Git에 강제 추가하지 마세요.